# Feature Engineering

## Modeling-Ready Report Usage Features

This notebook builds a clean feature layer from the processed semantic-model tables in `data/processed/`.

Notebook objective:
- Build the canonical daily report usage series using `build_report_daily_series`, anchored to each report's `launch_date` and `retire_date` from `dim_report`
- Add time-series usage features for rolling activity and week-over-week change
- Create separate behavioural and performance marts
- Join those marts into a modeling-ready `mart_forecast_features` table

## 1. Setup

We define project paths and import the reusable feature engineering helpers.


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.build_forecast_features import build_forecast_feature_table
from src.features.engagement_features import build_user_engagement_features
from src.features.performance_features import build_report_performance_features
from src.features.report_features import (
    add_time_series_usage_features,
    build_report_daily_adoption,
    build_report_daily_series,
)

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FACT_REPORT_VIEWS_PATH  = DATA_PROCESSED_DIR / "fact_report_views.csv"
FACT_PAGE_VIEWS_PATH    = DATA_PROCESSED_DIR / "fact_page_views.csv"
FACT_REPORT_LOADS_PATH  = DATA_PROCESSED_DIR / "fact_report_loads.csv"
DIM_REPORT_PATH         = DATA_PROCESSED_DIR / "dim_report.csv"

SERIES_OUTPUT_PATH      = DATA_PROCESSED_DIR / "mart_report_daily_series.csv"
OUTPUT_PATH             = DATA_PROCESSED_DIR / "mart_report_daily_adoption.csv"
TS_OUTPUT_PATH          = DATA_PROCESSED_DIR / "mart_report_daily_adoption_ts_features.csv"
ENGAGEMENT_OUTPUT_PATH  = DATA_PROCESSED_DIR / "mart_user_engagement.csv"
PERFORMANCE_OUTPUT_PATH = DATA_PROCESSED_DIR / "mart_report_performance.csv"
FORECAST_OUTPUT_PATH    = DATA_PROCESSED_DIR / "mart_forecast_features.csv"

## 2. Load Inputs

We load `fact_report_views` (event-level usage) and `dim_report` (report metadata, including `launch_date` and `retire_date`).

`dim_report` now carries `launch_date` and `retire_date` directly — promoted from `report_archetypes.csv` in the semantic-model build step — so this notebook does not need to reach back into `data/raw/` to determine active periods.

In [2]:
if not FACT_REPORT_VIEWS_PATH.exists():
    raise FileNotFoundError(f"Expected input file was not found: {FACT_REPORT_VIEWS_PATH}")
if not DIM_REPORT_PATH.exists():
    raise FileNotFoundError(f"Expected input file was not found: {DIM_REPORT_PATH}")

fact_report_views = pd.read_csv(FACT_REPORT_VIEWS_PATH)
dim_report = pd.read_csv(DIM_REPORT_PATH)

print(f"fact_report_views — loaded from: {FACT_REPORT_VIEWS_PATH}")
print("Shape:", fact_report_views.shape)
display(fact_report_views.head(3))

print(f"\ndim_report — loaded from: {DIM_REPORT_PATH}")
print("Shape:", dim_report.shape)
print("Columns:", dim_report.columns.tolist())
display(dim_report[["report_id", "archetype", "launch_date", "retire_date"]].head(8))

fact_report_views — loaded from: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/fact_report_views.csv
Shape: (135430, 6)


,date_key,report_id,user_key,consumption_method,distribution_method,view_count
0,20250101,R_001,UK_0183,Web,Direct,12
1,20250101,R_001,UK_0024,Mobile,Direct,1
2,20250101,R_001,UK_0180,Mobile,SharedLink,1



dim_report — loaded from: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/dim_report.csv
Shape: (30, 8)
Columns: ['report_id', 'report_name', 'workspace_id', 'report_type', 'is_usage_metrics_report', 'archetype', 'launch_date', 'retire_date']


,report_id,archetype,launch_date,retire_date
0,R_001,stable_weekly,2025-01-01,NaN
1,R_002,stable_weekly,2025-01-01,NaN
2,R_003,stable_weekly,2025-01-01,NaN
3,R_004,upward_adoption,2025-01-01,NaN
4,R_005,upward_adoption,2025-01-01,NaN
5,R_006,upward_adoption,2025-01-01,NaN
6,R_007,gradual_decline,2025-01-01,NaN
7,R_008,gradual_decline,2025-01-01,NaN


## 3. Build the Canonical Daily Series

`build_report_daily_series` produces one row for every (report, calendar date) pair within each report's active window, with `daily_views = 0` on days where no usage events occurred.

Active-period anchor — `launch_date` and `retire_date` from `dim_report`:
- Retired reports (R_010, R_011, R_012, R_025, R_026, R_027) end at their recorded retire date. The old inline approach silently extended them to the dataset end.
- Late-launch reports (R_022, R_023, R_024) start at their recorded launch date.
- Live reports with no retire date are active through the dataset-wide maximum observed date.

Two indicator columns make the source of every zero explicit:
- `is_observed_day` — at least one event existed in `fact_report_views` for this report-date.
- `is_imputed_zero` — the report was active but had no events; `daily_views` was filled with 0.

`unique_viewers` and `views_per_user` are then merged in from a separate `build_report_daily_adoption` call so the existing time-series and forecasting steps receive the same column structure they expect.

In [3]:
# Step 1 — canonical spine: one row per (report, date) within each report's active period
mart_report_daily_series = build_report_daily_series(
    fact_report_views=fact_report_views,
    report_active_periods=dim_report,
    date_col="date_key",
    report_col="report_id",
    views_col="view_count",
    active_start_col="launch_date",
    active_end_col="retire_date",
)

print("mart_report_daily_series")
print("  Shape:", mart_report_daily_series.shape)
print("  Active period source:", mart_report_daily_series.attrs["active_period_source"])
print("  is_observed_day True:", mart_report_daily_series["is_observed_day"].sum(),
      "| is_imputed_zero True:", mart_report_daily_series["is_imputed_zero"].sum())
display(mart_report_daily_series.head(4))

mart_report_daily_series
  Shape: (11847, 5)
  Active period source: report_active_periods
  is_observed_day True: 9518 | is_imputed_zero True: 2329


,report_id,date,daily_views,is_observed_day,is_imputed_zero
0,R_001,2025-01-01,39,True,False
1,R_001,2025-01-02,43,True,False
2,R_001,2025-01-03,35,True,False
3,R_001,2025-01-04,10,True,False


### Active-Period Spot Checks

Verify that retired and late-launch reports have correct date boundaries, and confirm imputed zeros appear in the right place.

In [4]:
# Retired report R_010 (retire_date 2025-09-12) — spine must end there
r10 = mart_report_daily_series[mart_report_daily_series["report_id"] == "R_010"]
print(f"R_010  rows: {len(r10)}  |  {r10['date'].min().date()} → {r10['date'].max().date()}")
display(r10.tail(4).to_string(index=False))

print()

# Late-launch R_024 (launch_date 2025-07-07) — spine must start there
r24 = mart_report_daily_series[mart_report_daily_series["report_id"] == "R_024"]
print(f"R_024  rows: {len(r24)}  |  {r24['date'].min().date()} → {r24['date'].max().date()}")
display(r24.head(4).to_string(index=False))

print()

# Step 2 — merge unique_viewers and views_per_user from the adoption aggregation
# (build_report_daily_adoption aggregates events; build_report_daily_series owns the spine)
report_daily_base = build_report_daily_adoption(
    fact_report_views=fact_report_views,
    date_col="date_key",
    report_col="report_id",
    user_col="user_key",
    views_col="view_count",
)

mart_report_daily_adoption = mart_report_daily_series.merge(
    report_daily_base[["date", "report_id", "unique_viewers", "views_per_user"]],
    on=["date", "report_id"],
    how="left",
)
mart_report_daily_adoption["unique_viewers"] = (
    mart_report_daily_adoption["unique_viewers"].fillna(0).astype(int)
)
mart_report_daily_adoption["views_per_user"] = (
    mart_report_daily_adoption["daily_views"]
    .div(mart_report_daily_adoption["unique_viewers"].replace(0, pd.NA))
    .fillna(0.0)
)

mart_report_daily_adoption = mart_report_daily_adoption[
    ["date", "report_id", "daily_views", "unique_viewers", "views_per_user",
     "is_observed_day", "is_imputed_zero"]
].sort_values(["report_id", "date"]).reset_index(drop=True)

print("mart_report_daily_adoption")
print("  Shape:", mart_report_daily_adoption.shape)
display(mart_report_daily_adoption.head(4))

R_010  rows: 255  |  2025-01-01 → 2025-09-12


'report_id       date  daily_views  is_observed_day  is_imputed_zero\n    R_010 2025-09-09           48             True            False\n    R_010 2025-09-10           43             True            False\n    R_010 2025-09-11           34             True            False\n    R_010 2025-09-12            0            False             True'


R_024  rows: 268  |  2025-07-07 → 2026-03-31


'report_id       date  daily_views  is_observed_day  is_imputed_zero\n    R_024 2025-07-07           62             True            False\n    R_024 2025-07-08           63             True            False\n    R_024 2025-07-09           86             True            False\n    R_024 2025-07-10           54             True            False'

mart_report_daily_adoption
  Shape: (11847, 7)


,date,report_id,daily_views,unique_viewers,views_per_user,is_observed_day,is_imputed_zero
0,2025-01-01,R_001,39,28,1.392857,True,False
1,2025-01-02,R_001,43,25,1.72,True,False
2,2025-01-03,R_001,35,24,1.458333,True,False
3,2025-01-04,R_001,10,7,1.428571,True,False


## 5. Validation Checks

These checks confirm the mart has the expected grain, boundaries align with `dim_report` active periods, no pre-launch rows exist, nulls are absent from key columns, and the indicator columns are internally consistent.

In [5]:
grain_is_unique = not mart_report_daily_adoption.duplicated(subset=["date", "report_id"]).any()

null_counts = mart_report_daily_adoption[
    ["date", "report_id", "daily_views", "unique_viewers", "views_per_user"]
].isnull().sum()

# Boundaries: each report must start on launch_date and end on min(retire_date, dataset_end)
active_bounds = dim_report[["report_id", "launch_date", "retire_date"]].copy()
active_bounds["launch_date"] = pd.to_datetime(active_bounds["launch_date"])
active_bounds["retire_date"] = pd.to_datetime(active_bounds["retire_date"])
dataset_end = mart_report_daily_adoption["date"].max()
active_bounds["retire_date"] = active_bounds["retire_date"].fillna(dataset_end)

report_date_ranges = mart_report_daily_adoption.groupby("report_id")["date"].agg(["min", "max"])
boundary_check = report_date_ranges.merge(active_bounds, on="report_id")
launch_aligned = (boundary_check["min"] == boundary_check["launch_date"]).all()
retire_aligned = (boundary_check["max"] == boundary_check["retire_date"]).all()

# Indicator consistency
observed_nonzero = (
    mart_report_daily_adoption.loc[mart_report_daily_adoption["is_observed_day"], "daily_views"].gt(0).all()
)
imputed_zero = (
    mart_report_daily_adoption.loc[mart_report_daily_adoption["is_imputed_zero"], "daily_views"].eq(0).all()
)
indicators_mutually_exclusive = (
    ~(mart_report_daily_adoption["is_observed_day"] & mart_report_daily_adoption["is_imputed_zero"])
).all()

zero_view_logic_holds = (
    mart_report_daily_adoption.loc[mart_report_daily_adoption["unique_viewers"] == 0, "views_per_user"]
    .eq(0).all()
)

print("One row per date/report_id:", grain_is_unique)
print("All reports start on launch_date:", launch_aligned)
print("All reports end on retire_date (or dataset end for live reports):", retire_aligned)
print("is_observed_day rows always have daily_views > 0:", observed_nonzero)
print("is_imputed_zero rows always have daily_views == 0:", imputed_zero)
print("is_observed_day and is_imputed_zero are mutually exclusive:", indicators_mutually_exclusive)
print("views_per_user is 0 when unique_viewers is 0:", zero_view_logic_holds)

print("\nNull counts in key columns:")
display(null_counts.to_frame(name="null_count"))

print("\nActive-period boundary check (first 8 reports):")
display(boundary_check[["report_id", "min", "max", "launch_date", "retire_date"]].head(8))

One row per date/report_id: True
All reports start on launch_date: True
All reports end on retire_date (or dataset end for live reports): True
is_observed_day rows always have daily_views > 0: True
is_imputed_zero rows always have daily_views == 0: True
is_observed_day and is_imputed_zero are mutually exclusive: True
views_per_user is 0 when unique_viewers is 0: True

Null counts in key columns:


,null_count
date,0
report_id,0
daily_views,0
unique_viewers,0
views_per_user,0



Active-period boundary check (first 8 reports):


,report_id,min,max,launch_date,retire_date
0,R_001,2025-01-01,2026-03-31,2025-01-01,2026-03-31
1,R_002,2025-01-01,2026-03-31,2025-01-01,2026-03-31
2,R_003,2025-01-01,2026-03-31,2025-01-01,2026-03-31
3,R_004,2025-01-01,2026-03-31,2025-01-01,2026-03-31
4,R_005,2025-01-01,2026-03-31,2025-01-01,2026-03-31
5,R_006,2025-01-01,2026-03-31,2025-01-01,2026-03-31
6,R_007,2025-01-01,2026-03-31,2025-01-01,2026-03-31
7,R_008,2025-01-01,2026-03-31,2025-01-01,2026-03-31


## 6. Summary Statistics

A quick distribution check helps confirm the current features look reasonable before saving the mart.


In [6]:
summary_stats = mart_report_daily_adoption[
    ["daily_views", "unique_viewers", "views_per_user"]
].describe()
display(summary_stats)


,daily_views,unique_viewers
count,11847.000000,11847.000000
mean,16.549253,10.920908
std,22.332323,15.181893
min,0.000000,0.000000
25%,2.000000,1.000000
50%,9.000000,5.000000
75%,24.000000,16.000000
max,369.000000,200.000000


## Time-Series Usage Features

Rolling metrics help us capture short-term and medium-term usage momentum for each report without changing the underlying daily grain. They are especially useful for spotting recent growth, softening demand, and stable recurring usage patterns.

A complete daily calendar matters because rolling windows should include true zero-usage days rather than skipping missing dates. That keeps the 7-day and 28-day features consistent across reports and makes later forecasting features more trustworthy.

Week-over-week change compares a report's current `daily_views` to its value 7 days earlier using `(current - lag_7d) / lag_7d`. When the 7-day lag is missing or zero, the result stays `NaN` so we avoid invalid infinite values.


In [7]:
mart_report_daily_adoption_ts = add_time_series_usage_features(mart_report_daily_adoption)

display(mart_report_daily_adoption_ts.head())
print("Shape:", mart_report_daily_adoption_ts.shape)


,date,report_id,daily_views,unique_viewers,views_per_user,is_observed_day,is_imputed_zero,views_7d,views_28d,viewers_7d,viewers_28d,wow_change_views
0,2025-01-01,R_001,39,28,1.392857,True,False,39.0,39.0,28.0,28.0,NaN
1,2025-01-02,R_001,43,25,1.72,True,False,82.0,82.0,53.0,53.0,NaN
2,2025-01-03,R_001,35,24,1.458333,True,False,117.0,117.0,77.0,77.0,NaN
3,2025-01-04,R_001,10,7,1.428571,True,False,127.0,127.0,84.0,84.0,NaN
4,2025-01-05,R_001,12,6,2.0,True,False,139.0,139.0,90.0,90.0,NaN


Shape: (11847, 12)


### Sample Report Time Series

Looking at a small sample of reports makes it easier to sanity-check the rolling features and week-over-week calculations.


In [8]:
sample_reports = mart_report_daily_adoption_ts["report_id"].drop_duplicates().head(2).tolist()
sample_time_series = mart_report_daily_adoption_ts.loc[
    mart_report_daily_adoption_ts["report_id"].isin(sample_reports),
    [
        "date",
        "report_id",
        "daily_views",
        "views_7d",
        "views_28d",
        "wow_change_views",
        "viewers_7d",
        "viewers_28d",
    ],
]
display(sample_time_series.head(20))


,date,report_id,daily_views,views_7d,views_28d,wow_change_views,viewers_7d,viewers_28d
0,2025-01-01,R_001,39,39.0,39.0,NaN,28.0,28.0
1,2025-01-02,R_001,43,82.0,82.0,NaN,53.0,53.0
2,2025-01-03,R_001,35,117.0,117.0,NaN,77.0,77.0
3,2025-01-04,R_001,10,127.0,127.0,NaN,84.0,84.0
4,2025-01-05,R_001,12,139.0,139.0,NaN,90.0,90.0
5,2025-01-06,R_001,56,195.0,195.0,NaN,122.0,122.0
6,2025-01-07,R_001,39,234.0,234.0,NaN,150.0,150.0
7,2025-01-08,R_001,40,235.0,274.0,0.025641,146.0,174.0
8,2025-01-09,R_001,31,223.0,305.0,-0.279070,142.0,195.0
9,2025-01-10,R_001,44,232.0,349.0,0.257143,144.0,221.0


### Time-Series Validation

These checks confirm that the time-series step enriches the existing dataset without changing row count or introducing invalid values.


In [9]:
row_count_unchanged = len(mart_report_daily_adoption_ts) == len(mart_report_daily_adoption)
ts_grain_is_unique = not mart_report_daily_adoption_ts.duplicated(subset=["date", "report_id"]).any()
rolling_null_counts = mart_report_daily_adoption_ts[["views_7d", "views_28d", "viewers_7d", "viewers_28d"]].isnull().sum()
rolling_columns_populated = rolling_null_counts.eq(0).all()
wow_has_infinite_values = mart_report_daily_adoption_ts["wow_change_views"].isin([float("inf"), float("-inf")]).any()
ts_summary_stats = mart_report_daily_adoption_ts[["views_7d", "views_28d", "wow_change_views"]].describe()
wow_nan_examples = mart_report_daily_adoption_ts.loc[
    mart_report_daily_adoption_ts["wow_change_views"].isna(),
    ["date", "report_id", "daily_views", "views_7d", "views_28d", "wow_change_views"],
].head(10)

print("Row count unchanged:", row_count_unchanged)
print("One row per date/report_id:", ts_grain_is_unique)
print("Rolling columns populated:", rolling_columns_populated)
print("\nNull counts for rolling columns:")
display(rolling_null_counts.to_frame(name="null_count"))
print("wow_change_views contains no infinite values:", not wow_has_infinite_values)

print("\nSummary statistics:")
display(ts_summary_stats)

print("Example rows where wow_change_views is NaN:")
display(wow_nan_examples)


Row count unchanged: True
One row per date/report_id: True
Rolling columns populated: True

Null counts for rolling columns:


,null_count
views_7d,0
views_28d,0
viewers_7d,0
viewers_28d,0


wow_change_views contains no infinite values: True

Summary statistics:


,views_7d,views_28d,wow_change_views
count,11847.000000,11847.000000,9354.000000
mean,114.214738,441.496159,0.450033
std,102.448305,366.150839,4.655991
min,0.000000,0.000000,-1.000000
25%,33.000000,175.000000,-0.272727
50%,88.000000,366.000000,0.000000
75%,181.000000,638.000000,0.310345
max,703.000000,2569.000000,208.000000


Example rows where wow_change_views is NaN:


,date,report_id,daily_views,views_7d,views_28d,wow_change_views
0,2025-01-01,R_001,39,39.0,39.0,NaN
1,2025-01-02,R_001,43,82.0,82.0,NaN
2,2025-01-03,R_001,35,117.0,117.0,NaN
3,2025-01-04,R_001,10,127.0,127.0,NaN
4,2025-01-05,R_001,12,139.0,139.0,NaN
5,2025-01-06,R_001,56,195.0,195.0,NaN
6,2025-01-07,R_001,39,234.0,234.0,NaN
36,2025-02-06,R_001,49,157.0,887.0,NaN
37,2025-02-07,R_001,54,211.0,897.0,NaN
40,2025-02-10,R_001,45,260.0,909.0,NaN


### Save `mart_report_daily_series` and `mart_report_daily_adoption_ts_features`

`mart_report_daily_series` is the canonical active-period series with indicator columns. The TS-enriched adoption mart is saved separately for downstream feature assembly.

In [10]:
mart_report_daily_series.to_csv(SERIES_OUTPUT_PATH, index=False)
mart_report_daily_adoption_ts.to_csv(TS_OUTPUT_PATH, index=False)
print(f"Saved mart_report_daily_series to:        {SERIES_OUTPUT_PATH}")
print(f"Saved mart_report_daily_adoption_ts to:   {TS_OUTPUT_PATH}")

Saved mart_report_daily_series to:        /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/mart_report_daily_series.csv
Saved mart_report_daily_adoption_ts to:   /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/mart_report_daily_adoption_ts_features.csv


## Behavioural Features

Usage features tell us how much activity a report receives. Behavioural features tell us how people use that report, including whether users return, whether a small group dominates usage, how long a report sits idle between active days, and how deep people navigate into pages.

These metrics matter because similar view totals can mask very different engagement patterns. A high repeat user rate suggests recurring habits, a high concentration share suggests dependence on a small group of power users, and page-depth helps separate quick glances from deeper exploration.

This section builds:
- `repeat_user_rate`
- `top_10pct_user_share`
- `days_since_last_use`
- `avg_pages_per_user`


### Load Behavioural Inputs

We reuse `fact_report_views` and load `fact_page_views` from the processed layer. In this repo, `date_key` and `user_key` play the role of the generic `date` and `user_id` columns.


In [11]:
if not FACT_PAGE_VIEWS_PATH.exists():
    raise FileNotFoundError(f"Expected input file was not found: {FACT_PAGE_VIEWS_PATH}")

fact_page_views = pd.read_csv(FACT_PAGE_VIEWS_PATH)
print(f"Loaded page views from: {FACT_PAGE_VIEWS_PATH}")
print("Shape:", fact_page_views.shape)
display(fact_page_views.head())


Loaded page views from: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/fact_page_views.csv
Shape: (270635, 7)


,date_key,report_id,page_key,user_key,client,session_source,page_view_count
0,20250101,R_001,3,UK_0183,Browser,App,1
1,20250101,R_001,1,UK_0183,Browser,Direct,1
2,20250101,R_001,2,UK_0183,Browser,App,1
3,20250101,R_001,4,UK_0183,Browser,Direct,1
4,20250101,R_001,5,UK_0183,Browser,App,1


### Build `mart_user_engagement`

The engagement mart stays separate from `mart_report_daily_adoption` and focuses only on behavioural signals. For the page-depth proxy, the processed page-view fact uses `page_key` instead of `section_id`, which is sufficient for counting per-user page activity.


In [12]:
mart_user_engagement = build_user_engagement_features(
    fact_report_views=fact_report_views,
    fact_page_views=fact_page_views,
    date_col="date_key",
    report_col="report_id",
    user_col="user_key",
)

display(mart_user_engagement.head())
print("Shape:", mart_user_engagement.shape)


,date,report_id,repeat_user_rate,top_10pct_user_share,days_since_last_use,avg_pages_per_user
0,2025-01-01,R_001,0.000000,0.358974,0,1.928571
1,2025-01-02,R_001,0.240000,0.488372,1,1.640000
2,2025-01-03,R_001,0.458333,0.400000,1,2.166667
3,2025-01-04,R_001,0.714286,0.400000,1,2.428571
4,2025-01-05,R_001,0.833333,0.416667,1,2.666667


Shape: (10212, 6)


### Behavioural Validation

These checks confirm the mart stays at the expected grain and that each behavioural feature remains within a sensible range.


In [13]:
engagement_grain_is_unique = not mart_user_engagement.duplicated(subset=["date", "report_id"]).any()
engagement_null_counts = mart_user_engagement[
    [
        "date",
        "report_id",
        "repeat_user_rate",
        "top_10pct_user_share",
        "days_since_last_use",
        "avg_pages_per_user",
    ]
].isnull().sum()
repeat_rate_in_range = mart_user_engagement["repeat_user_rate"].between(0, 1).all()
top_share_in_range = mart_user_engagement["top_10pct_user_share"].between(0, 1).all()
days_since_non_negative = mart_user_engagement["days_since_last_use"].ge(0).all()
avg_pages_non_negative = mart_user_engagement["avg_pages_per_user"].ge(0).all()

print("One row per date/report_id:", engagement_grain_is_unique)
print("No duplicate keys:", engagement_grain_is_unique)
print("No nulls in key columns:", engagement_null_counts[["date", "report_id"]].eq(0).all())
print("repeat_user_rate between 0 and 1:", repeat_rate_in_range)
print("top_10pct_user_share between 0 and 1:", top_share_in_range)
print("days_since_last_use >= 0:", days_since_non_negative)
print("avg_pages_per_user >= 0:", avg_pages_non_negative)

print("\nNull counts:")
display(engagement_null_counts.to_frame(name="null_count"))


One row per date/report_id: True
No duplicate keys: True
No nulls in key columns: True
repeat_user_rate between 0 and 1: True
top_10pct_user_share between 0 and 1: True
days_since_last_use >= 0: True
avg_pages_per_user >= 0: True

Null counts:


,null_count
date,0
report_id,0
repeat_user_rate,0
top_10pct_user_share,0
days_since_last_use,0
avg_pages_per_user,0


### Behavioural Examples

We inspect a few reports over time and highlight example rows where repeat usage changes, concentration is high, and the gap since last use becomes larger.


In [14]:
example_reports = mart_user_engagement["report_id"].drop_duplicates().head(3).tolist()
engagement_examples = mart_user_engagement.loc[
    mart_user_engagement["report_id"].isin(example_reports),
    [
        "date",
        "report_id",
        "repeat_user_rate",
        "top_10pct_user_share",
        "days_since_last_use",
        "avg_pages_per_user",
    ],
]
display(engagement_examples.head(30))

repeat_rate_changes = mart_user_engagement.groupby("report_id")["repeat_user_rate"].diff().fillna(0).ne(0)
changing_repeat_examples = mart_user_engagement.loc[
    repeat_rate_changes,
    ["date", "report_id", "repeat_user_rate"],
].head(10)
high_concentration_examples = mart_user_engagement.loc[
    mart_user_engagement["top_10pct_user_share"] >= 0.5,
    ["date", "report_id", "top_10pct_user_share", "repeat_user_rate"],
].head(10)
days_since_spike_examples = mart_user_engagement.loc[
    mart_user_engagement["days_since_last_use"] >= 2,
    ["date", "report_id", "days_since_last_use", "repeat_user_rate"],
].head(10)

print("Example rows where repeat_user_rate changes:")
display(changing_repeat_examples)

print("Example rows with high concentration:")
display(high_concentration_examples)

print("Example rows where days_since_last_use spikes:")
display(days_since_spike_examples)


,date,report_id,repeat_user_rate,top_10pct_user_share,days_since_last_use,avg_pages_per_user
0,2025-01-01,R_001,0.000000,0.358974,0,1.928571
1,2025-01-02,R_001,0.240000,0.488372,1,1.640000
2,2025-01-03,R_001,0.458333,0.400000,1,2.166667
3,2025-01-04,R_001,0.714286,0.400000,1,2.428571
4,2025-01-05,R_001,0.833333,0.416667,1,2.666667
5,2025-01-06,R_001,0.562500,0.500000,1,1.937500
6,2025-01-07,R_001,0.571429,0.333333,1,1.928571
7,2025-01-08,R_001,0.750000,0.475000,1,2.666667
8,2025-01-09,R_001,0.809524,0.419355,1,1.952381
9,2025-01-10,R_001,0.807692,0.454545,1,2.153846


Example rows where repeat_user_rate changes:


,date,report_id,repeat_user_rate
1,2025-01-02,R_001,0.240000
2,2025-01-03,R_001,0.458333
3,2025-01-04,R_001,0.714286
4,2025-01-05,R_001,0.833333
5,2025-01-06,R_001,0.562500
6,2025-01-07,R_001,0.571429
7,2025-01-08,R_001,0.750000
8,2025-01-09,R_001,0.809524
9,2025-01-10,R_001,0.807692
10,2025-01-11,R_001,0.857143


Example rows with high concentration:


,date,report_id,top_10pct_user_share,repeat_user_rate
5,2025-01-06,R_001,0.500000,0.562500
25,2025-01-26,R_001,0.666667,1.000000
29,2025-02-01,R_001,0.555556,1.000000
45,2025-02-18,R_001,0.512821,1.000000
46,2025-02-19,R_001,0.513514,0.947368
50,2025-02-23,R_001,0.625000,1.000000
68,2025-03-19,R_001,0.513514,1.000000
71,2025-03-22,R_001,0.666667,1.000000
80,2025-03-31,R_001,0.550000,0.952381
85,2025-04-05,R_001,0.500000,1.000000


Example rows where days_since_last_use spikes:


,date,report_id,days_since_last_use,repeat_user_rate
29,2025-02-01,R_001,3,1.000000
31,2025-02-04,R_001,2,0.911765
53,2025-03-03,R_001,6,1.000000
61,2025-03-12,R_001,2,0.944444
97,2025-04-22,R_001,6,1.000000
126,2025-05-22,R_001,2,1.000000
283,2025-10-29,R_001,4,1.000000
385,2026-02-13,R_001,6,1.000000
450,2025-01-24,R_002,6,0.818182
456,2025-02-01,R_002,3,0.800000


### Save `mart_user_engagement`

The behavioural mart is saved separately so it can evolve independently from the adoption and time-series outputs.


In [15]:
mart_user_engagement.to_csv(ENGAGEMENT_OUTPUT_PATH, index=False)
print(f"Saved mart_user_engagement to: {ENGAGEMENT_OUTPUT_PATH}")


Saved mart_user_engagement to: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/mart_user_engagement.csv


## Performance Features

Performance telemetry matters because slower reports can shape adoption, repeat usage, and overall user experience even when the content itself is valuable. If reports load slowly or inconsistently, users may open them less often, abandon them sooner, or rely on only a small set of trusted reports.

This section builds:
- `avg_load_time`: average load time for each report-day
- `p90_load_time`: upper-end latency for the same report-day
- `avg_load_time_7d`: trailing 7-day rolling average of average load time
- `load_time_wow_change`: percentage change in average load time compared with 7 days earlier


### Load Performance Input

We load `fact_report_loads` from the processed layer. In this repo, the date field is stored as `date_key` and the load-time measure is stored as `load_time_ms`.


In [16]:
if not FACT_REPORT_LOADS_PATH.exists():
    raise FileNotFoundError(f"Expected input file was not found: {FACT_REPORT_LOADS_PATH}")

fact_report_loads = pd.read_csv(FACT_REPORT_LOADS_PATH)
print(f"Loaded report loads from: {FACT_REPORT_LOADS_PATH}")
print("Shape:", fact_report_loads.shape)
display(fact_report_loads.head())


Loaded report loads from: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/fact_report_loads.csv
Shape: (135430, 7)


,date_key,report_id,user_key,browser,client,country,load_time_ms
0,20250101,R_001,UK_0183,Chrome,Browser,Canada,4009.0
1,20250101,R_001,UK_0024,Edge,MobileApp,UK,3843.0
2,20250101,R_001,UK_0180,Chrome,MobileApp,Canada,2900.0
3,20250101,R_001,UK_0013,Edge,Browser,UK,3063.0
4,20250101,R_001,UK_0032,Edge,Browser,Canada,2540.0


### Build `mart_report_performance`

The performance mart stays separate from the usage and behavioural marts. We aggregate load events to the `date` and `report_id` grain and then add simple rolling and week-over-week performance signals.


In [17]:
mart_report_performance = build_report_performance_features(
    fact_report_loads=fact_report_loads,
    date_col="date_key",
    report_col="report_id",
    load_time_col="load_time_ms",
)

display(mart_report_performance.head())
print("Shape:", mart_report_performance.shape)


,date,report_id,avg_load_time,p90_load_time,avg_load_time_7d,load_time_wow_change,load_events
0,2025-01-01,R_001,3509.428571,4045.9,3509.428571,NaN,28
1,2025-01-02,R_001,3637.440000,4317.6,3573.434286,NaN,25
2,2025-01-03,R_001,3402.625000,3768.7,3516.497857,NaN,24
3,2025-01-04,R_001,3253.714286,3770.8,3450.801964,NaN,7
4,2025-01-05,R_001,3391.000000,3604.0,3438.841571,NaN,6


Shape: (10212, 7)


### Performance Validation

These checks confirm the performance mart stays at one row per report-day and that the engineered measures remain numerically sensible.


In [18]:
performance_grain_is_unique = not mart_report_performance.duplicated(subset=["date", "report_id"]).any()
performance_null_counts = mart_report_performance[
    ["date", "report_id", "avg_load_time", "p90_load_time", "avg_load_time_7d", "load_time_wow_change"]
].isnull().sum()
avg_load_time_non_negative = mart_report_performance["avg_load_time"].ge(0).all()
p90_load_time_non_negative = mart_report_performance["p90_load_time"].ge(0).all()
multiple_event_mask = mart_report_performance["load_events"] > 1
p90_at_least_avg_when_multiple_events = mart_report_performance.loc[
    multiple_event_mask, "p90_load_time"
].ge(mart_report_performance.loc[multiple_event_mask, "avg_load_time"]).all()
avg_load_time_7d_non_negative = mart_report_performance["avg_load_time_7d"].ge(0).all()
load_time_wow_has_infinite_values = mart_report_performance["load_time_wow_change"].isin([float("inf"), float("-inf")]).any()

print("One row per date/report_id:", performance_grain_is_unique)
print("No duplicate keys:", performance_grain_is_unique)
print("No nulls in key columns:", performance_null_counts[["date", "report_id"]].eq(0).all())
print("avg_load_time >= 0:", avg_load_time_non_negative)
print("p90_load_time >= 0:", p90_load_time_non_negative)
print("p90_load_time >= avg_load_time when multiple events exist:", p90_at_least_avg_when_multiple_events)
print("avg_load_time_7d >= 0:", avg_load_time_7d_non_negative)
print("load_time_wow_change contains no infinite values:", not load_time_wow_has_infinite_values)

print("\nNull counts:")
display(performance_null_counts.to_frame(name="null_count"))


One row per date/report_id: True
No duplicate keys: True
No nulls in key columns: True
avg_load_time >= 0: True
p90_load_time >= 0: True
p90_load_time >= avg_load_time when multiple events exist: True
avg_load_time_7d >= 0: True
load_time_wow_change contains no infinite values: True

Null counts:


,null_count
date,0
report_id,0
avg_load_time,0
p90_load_time,0
avg_load_time_7d,0
load_time_wow_change,210


### Performance Examples

We inspect a few reports over time and show a simple comparison of load time versus usage for the same report-day where the adoption mart is already available in the notebook.


In [19]:
performance_example_reports = mart_report_performance["report_id"].drop_duplicates().head(3).tolist()
performance_examples = mart_report_performance.loc[
    mart_report_performance["report_id"].isin(performance_example_reports),
    ["date", "report_id", "avg_load_time", "p90_load_time", "avg_load_time_7d", "load_time_wow_change"],
]
display(performance_examples.head(30))

performance_summary_stats = mart_report_performance[
    ["avg_load_time", "p90_load_time", "avg_load_time_7d", "load_time_wow_change"]
].describe()
print("Summary statistics:")
display(performance_summary_stats)

load_time_vs_usage_examples = mart_report_performance.merge(
    mart_report_daily_adoption[["date", "report_id", "daily_views"]],
    on=["date", "report_id"],
    how="left",
)
load_time_vs_usage_examples = load_time_vs_usage_examples.loc[
    load_time_vs_usage_examples["report_id"].isin(performance_example_reports),
    ["date", "report_id", "avg_load_time", "p90_load_time", "daily_views"],
]
print("Example comparison of load time vs usage:")
display(load_time_vs_usage_examples.head(30))


,date,report_id,avg_load_time,p90_load_time,avg_load_time_7d,load_time_wow_change
0,2025-01-01,R_001,3509.428571,4045.9,3509.428571,NaN
1,2025-01-02,R_001,3637.440000,4317.6,3573.434286,NaN
2,2025-01-03,R_001,3402.625000,3768.7,3516.497857,NaN
3,2025-01-04,R_001,3253.714286,3770.8,3450.801964,NaN
4,2025-01-05,R_001,3391.000000,3604.0,3438.841571,NaN
5,2025-01-06,R_001,3474.687500,3866.3,3444.815893,NaN
6,2025-01-07,R_001,3463.821429,3918.6,3447.530969,NaN
7,2025-01-08,R_001,3526.708333,4067.7,3449.999507,0.004924
8,2025-01-09,R_001,3586.571429,4282.0,3442.732568,-0.013985
9,2025-01-10,R_001,3461.307692,4019.5,3451.115810,0.017246


Summary statistics:


,avg_load_time,p90_load_time,avg_load_time_7d,load_time_wow_change
count,10212.000000,10212.000000,10212.000000,10002.000000
mean,3648.284953,4081.806414,3647.992795,0.006680
std,1211.996965,1203.597680,1188.940689,0.119316
min,500.000000,500.000000,912.300758,-0.694842
25%,2979.653010,3348.675000,3048.598385,-0.048742
50%,3551.006944,4028.000000,3557.864608,-0.000661
75%,4166.593750,4608.850000,4134.630952,0.051903
max,7752.000000,8165.300000,6859.067857,2.001550


Example comparison of load time vs usage:


,date,report_id,avg_load_time,p90_load_time,daily_views
0,2025-01-01,R_001,3509.428571,4045.9,39.0
1,2025-01-02,R_001,3637.440000,4317.6,43.0
2,2025-01-03,R_001,3402.625000,3768.7,35.0
3,2025-01-04,R_001,3253.714286,3770.8,10.0
4,2025-01-05,R_001,3391.000000,3604.0,12.0
5,2025-01-06,R_001,3474.687500,3866.3,56.0
6,2025-01-07,R_001,3463.821429,3918.6,39.0
7,2025-01-08,R_001,3526.708333,4067.7,40.0
8,2025-01-09,R_001,3586.571429,4282.0,31.0
9,2025-01-10,R_001,3461.307692,4019.5,44.0


### Save `mart_report_performance`

The performance mart is saved separately so it can evolve independently from the usage and behavioural marts.


In [20]:
mart_report_performance.to_csv(PERFORMANCE_OUTPUT_PATH, index=False)
print(f"Saved mart_report_performance to: {PERFORMANCE_OUTPUT_PATH}")


Saved mart_report_performance to: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/mart_report_performance.csv


## 7. Final Forecasting Feature Table

The final modeling table starts from the adoption mart, then appends behavioural and performance features with a left join on `date` and `report_id`. In the joined output, the behavioural concentration feature uses the clearer name `top_user_share`, which maps to the underlying `top_10pct_user_share` logic from the engagement mart.


In [21]:
mart_forecast_features = build_forecast_feature_table(
    mart_report_daily_adoption=mart_report_daily_adoption_ts,
    mart_user_engagement=mart_user_engagement,
    mart_report_performance=mart_report_performance,
)

print("Shape:", mart_forecast_features.shape)
display(mart_forecast_features.head())
print("Columns:")
display(pd.Index(mart_forecast_features.columns))


Shape: (11847, 21)


,date,report_id,daily_views,unique_viewers,views_per_user,is_observed_day,is_imputed_zero,views_7d,views_28d,viewers_7d,...,wow_change_views,repeat_user_rate,top_user_share,days_since_last_use,avg_pages_per_user,avg_load_time,p90_load_time,avg_load_time_7d,load_time_wow_change,load_events
0,2025-01-01,R_001,39,28,1.392857,True,False,39.0,39.0,28.0,...,NaN,0.000000,0.358974,0.0,1.928571,3509.428571,4045.9,3509.428571,NaN,28.0
1,2025-01-02,R_001,43,25,1.72,True,False,82.0,82.0,53.0,...,NaN,0.240000,0.488372,1.0,1.640000,3637.440000,4317.6,3573.434286,NaN,25.0
2,2025-01-03,R_001,35,24,1.458333,True,False,117.0,117.0,77.0,...,NaN,0.458333,0.400000,1.0,2.166667,3402.625000,3768.7,3516.497857,NaN,24.0
3,2025-01-04,R_001,10,7,1.428571,True,False,127.0,127.0,84.0,...,NaN,0.714286,0.400000,1.0,2.428571,3253.714286,3770.8,3450.801964,NaN,7.0
4,2025-01-05,R_001,12,6,2.0,True,False,139.0,139.0,90.0,...,NaN,0.833333,0.416667,1.0,2.666667,3391.000000,3604.0,3438.841571,NaN,6.0


Columns:


Index(['date', 'report_id', 'daily_views', 'unique_viewers', 'views_per_user',
       'is_observed_day', 'is_imputed_zero', 'views_7d', 'views_28d',
       'viewers_7d', 'viewers_28d', 'wow_change_views', 'repeat_user_rate',
       'top_user_share', 'days_since_last_use', 'avg_pages_per_user',
       'avg_load_time', 'p90_load_time', 'avg_load_time_7d',
       'load_time_wow_change', 'load_events'],
      dtype='str')

## 8. Validation and Preview

These final checks confirm that the joined mart stays unique at the report-day grain, contains the expected core features, and keeps the most important numeric fields within sensible ranges.


In [22]:
expected_core_columns = [
    "date",
    "report_id",
    "daily_views",
    "unique_viewers",
    "views_per_user",
    "views_7d",
    "views_28d",
    "wow_change_views",
    "repeat_user_rate",
    "top_user_share",
    "days_since_last_use",
    "avg_pages_per_user",
    "avg_load_time",
    "p90_load_time",
    "avg_load_time_7d",
    "load_time_wow_change",
]
missing_core_columns = [column for column in expected_core_columns if column not in mart_forecast_features.columns]
forecast_grain_is_unique = not mart_forecast_features.duplicated(subset=["date", "report_id"]).any()
forecast_null_counts = mart_forecast_features[["date", "report_id"]].isnull().sum()
daily_views_non_negative = mart_forecast_features["daily_views"].ge(0).all()
unique_viewers_non_negative = mart_forecast_features["unique_viewers"].ge(0).all()
repeat_user_rate_valid = mart_forecast_features["repeat_user_rate"].dropna().between(0, 1).all()
top_user_share_valid = mart_forecast_features["top_user_share"].dropna().between(0, 1).all()
avg_pages_non_negative = mart_forecast_features["avg_pages_per_user"].dropna().ge(0).all()
avg_load_time_non_negative = mart_forecast_features["avg_load_time"].dropna().ge(0).all()
p90_load_time_non_negative = mart_forecast_features["p90_load_time"].dropna().ge(0).all()
wow_change_has_inf = mart_forecast_features["wow_change_views"].isin([float("inf"), float("-inf")]).any()
load_time_wow_has_inf = mart_forecast_features["load_time_wow_change"].isin([float("inf"), float("-inf")]).any()

print("One row per date/report_id:", forecast_grain_is_unique)
print("No duplicate keys:", forecast_grain_is_unique)
print("No nulls in key columns:", forecast_null_counts.eq(0).all())
print("Expected core columns exist:", len(missing_core_columns) == 0)
print("daily_views >= 0:", daily_views_non_negative)
print("unique_viewers >= 0:", unique_viewers_non_negative)
print("repeat_user_rate between 0 and 1 where present:", repeat_user_rate_valid)
print("top_user_share between 0 and 1 where present:", top_user_share_valid)
print("avg_pages_per_user >= 0 where present:", avg_pages_non_negative)
print("avg_load_time >= 0 where present:", avg_load_time_non_negative)
print("p90_load_time >= 0 where present:", p90_load_time_non_negative)
print("wow_change_views contains no infinite values:", not wow_change_has_inf)
print("load_time_wow_change contains no infinite values:", not load_time_wow_has_inf)

if missing_core_columns:
    print("Missing core columns:", missing_core_columns)


One row per date/report_id: True
No duplicate keys: True
No nulls in key columns: True
Expected core columns exist: True
daily_views >= 0: True
unique_viewers >= 0: True
repeat_user_rate between 0 and 1 where present: True
top_user_share between 0 and 1 where present: True
avg_pages_per_user >= 0 where present: True
avg_load_time >= 0 where present: True
p90_load_time >= 0 where present: True
wow_change_views contains no infinite values: True
load_time_wow_change contains no infinite values: True


In [23]:
forecast_summary = pd.DataFrame(
    {
        "metric": ["row_count", "distinct_reports", "date_min", "date_max"],
        "value": [
            len(mart_forecast_features),
            mart_forecast_features["report_id"].nunique(),
            mart_forecast_features["date"].min(),
            mart_forecast_features["date"].max(),
        ],
    }
)
key_feature_missingness = mart_forecast_features[
    [
        "daily_views",
        "unique_viewers",
        "repeat_user_rate",
        "top_user_share",
        "avg_pages_per_user",
        "avg_load_time",
        "p90_load_time",
        "wow_change_views",
        "load_time_wow_change",
    ]
].isnull().sum().to_frame(name="missing_count")

print("Final mart summary:")
display(forecast_summary)

print("Missingness summary for key feature columns:")
display(key_feature_missingness)


Final mart summary:


,metric,value
0,row_count,11847
1,distinct_reports,30
2,date_min,2025-01-01 00:00:00
3,date_max,2026-03-31 00:00:00


Missingness summary for key feature columns:


,missing_count
daily_views,0
unique_viewers,0
repeat_user_rate,2329
top_user_share,2329
avg_pages_per_user,2329
avg_load_time,2329
p90_load_time,2329
wow_change_views,2493
load_time_wow_change,2539


In [24]:
forecast_example_reports = mart_forecast_features["report_id"].drop_duplicates().head(3).tolist()
forecast_example_histories = mart_forecast_features.loc[
    mart_forecast_features["report_id"].isin(forecast_example_reports),
    [
        "date",
        "report_id",
        "daily_views",
        "views_7d",
        "repeat_user_rate",
        "top_user_share",
        "avg_pages_per_user",
        "avg_load_time",
        "load_time_wow_change",
    ],
]
display(forecast_example_histories.head(30))


,date,report_id,daily_views,views_7d,repeat_user_rate,top_user_share,avg_pages_per_user,avg_load_time,load_time_wow_change
0,2025-01-01,R_001,39,39.0,0.000000,0.358974,1.928571,3509.428571,NaN
1,2025-01-02,R_001,43,82.0,0.240000,0.488372,1.640000,3637.440000,NaN
2,2025-01-03,R_001,35,117.0,0.458333,0.400000,2.166667,3402.625000,NaN
3,2025-01-04,R_001,10,127.0,0.714286,0.400000,2.428571,3253.714286,NaN
4,2025-01-05,R_001,12,139.0,0.833333,0.416667,2.666667,3391.000000,NaN
5,2025-01-06,R_001,56,195.0,0.562500,0.500000,1.937500,3474.687500,NaN
6,2025-01-07,R_001,39,234.0,0.571429,0.333333,1.928571,3463.821429,NaN
7,2025-01-08,R_001,40,235.0,0.750000,0.475000,2.666667,3526.708333,0.004924
8,2025-01-09,R_001,31,223.0,0.809524,0.419355,1.952381,3586.571429,-0.013985
9,2025-01-10,R_001,44,232.0,0.807692,0.454545,2.153846,3461.307692,0.017246


## 9. Save Outputs

We save the marts together so the processed feature layer stays coherent and easy to reuse in later forecasting work.


In [25]:
mart_report_daily_series.to_csv(SERIES_OUTPUT_PATH, index=False)
mart_report_daily_adoption.to_csv(OUTPUT_PATH, index=False)
mart_user_engagement.to_csv(ENGAGEMENT_OUTPUT_PATH, index=False)
mart_report_performance.to_csv(PERFORMANCE_OUTPUT_PATH, index=False)
mart_forecast_features.to_csv(FORECAST_OUTPUT_PATH, index=False)

print(f"Saved mart_report_daily_series to:    {SERIES_OUTPUT_PATH}")
print(f"Saved mart_report_daily_adoption to:  {OUTPUT_PATH}")
print(f"Saved mart_user_engagement to:        {ENGAGEMENT_OUTPUT_PATH}")
print(f"Saved mart_report_performance to:     {PERFORMANCE_OUTPUT_PATH}")
print(f"Saved mart_forecast_features to:      {FORECAST_OUTPUT_PATH}")

Saved mart_report_daily_series to:    /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/mart_report_daily_series.csv
Saved mart_report_daily_adoption to:  /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/mart_report_daily_adoption.csv
Saved mart_user_engagement to:        /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/mart_user_engagement.csv
Saved mart_report_performance to:     /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting/data/processed/mart_report_performance.csv
Saved mart_forecast_features to:      /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Versi

## 10. Summary and Next Step

This notebook now produces four connected outputs: adoption and usage features, behavioural features, performance features, and the final joined `mart_forecast_features` table. The next step is to use `mart_forecast_features` as the modeling-ready input for report-level forecasting experiments and evaluation.
